# Proyecto Integrador — Sismos en Perú
**Curso:** Lenguaje de Programación II
**Tema:** Análisis de la actividad sísmica reciente en Perú usando la API pública del USGS

**Requisitos obligatorios cubiertos en este notebook:**
1. Programación Orientada a Objetos (3 clases: `ExtractorSismos`, `ProcesadorSismos`, `VisualizadorSismos`)
2. Programas en red (uso de `requests` para HTTP)
3. Expresiones regulares (extracción de patrón en el texto del lugar del sismo)
4. API con información real y actualizada (USGS Earthquake API)
5. Procesamiento con pandas (limpieza, transformación, análisis)
6. Visualización (3 gráficos distintos)


## 2. Clase `ExtractorSismos` (Programas en red / API)

Esta clase se conecta a la **API pública del USGS** (`https://earthquake.usgs.gov`), que es
gratuita, no requiere API key y entrega datos reales y actualizados de sismos en formato GeoJSON.
Se filtra solo el territorio peruano usando un *bounding box* de coordenadas.
Aquí se cumple el requisito de **Programas en red** (uso de `requests` para HTTP GET).

In [1]:
# APORTE INTEGRANTE 2: Clase ExtractorSismos (conexión de red)
class ExtractorSismos:
    """Clase encargada de la conexión de red y obtención de datos de sismos en Perú."""

    LAT_MIN, LAT_MAX = -18.5, -0.0
    LON_MIN, LON_MAX = -81.5, -68.5

    def __init__(self, dias_atras: int = 90, magnitud_minima: float = 2.5):
        self.url_base = "https://earthquake.usgs.gov/fdsnws/event/1/query"
        self.dias_atras = dias_atras
        self.magnitud_minima = magnitud_minima
        self.datos_brutos = None

    def conectar_y_descargar(self) -> dict:
        """Realiza una solicitud HTTP GET a la API del USGS y descarga los sismos en Perú."""
        fecha_fin = datetime.utcnow()
        fecha_inicio = fecha_fin - timedelta(days=self.dias_atras)

        parametros = {
            "format": "geojson",
            "starttime": fecha_inicio.strftime("%Y-%m-%d"),
            "endtime": fecha_fin.strftime("%Y-%m-%d"),
            "minlatitude": self.LAT_MIN,
            "maxlatitude": self.LAT_MAX,
            "minlongitude": self.LON_MIN,
            "maxlongitude": self.LON_MAX,
            "minmagnitude": self.magnitud_minima,
            "orderby": "time",
        }

        try:
            print(f"[INFO] Conectando a la red: {self.url_base}")
            respuesta = requests.get(self.url_base, params=parametros, timeout=15)

            if respuesta.status_code == 200:
                self.datos_brutos = respuesta.json()
                total = len(self.datos_brutos.get("features", []))
                print(f"[INFO] Datos descargados correctamente. Sismos encontrados: {total}")
                return self.datos_brutos
            else:
                print(f"[ERROR] Código de estado HTTP inesperado: {respuesta.status_code}")
                return {}
        except requests.exceptions.RequestException as e:
            print(f"[CRÍTICO] Fallo en la conexión de red: {e}")
            return {}

# Ejecutamos la descarga real
extractor = ExtractorSismos(dias_atras=DIAS_A_CONSULTAR, magnitud_minima=MAGNITUD_MINIMA)
datos_crudos = extractor.conectar_y_descargar()


NameError: name 'DIAS_A_CONSULTAR' is not defined

## 3. Clase `ProcesadorSismos` (Procesamiento con Pandas / Expresiones Regulares)

Esta clase constituye el núcleo metodológico del proyecto. Se encarga de recibir los datos brutos en formato GeoJSON (descargados por la clase `ExtractorSismos`), limpiarlos de registros incompletos, estructurar las variables numéricas y aplicar técnicas avanzadas de parsing de texto para la minería de datos geoespaciales.

Aquí se cumplen rigurosamente dos de los requisitos obligatorios del proyecto:
1. **Expresiones Regulares (Regex):** Extracción precisa de distancias, orientaciones y ciudades de referencia codificadas en la cadena de texto original del sismo.
2. **Procesamiento de datos con Pandas:** Construcción de un objeto `DataFrame`, filtrado de datos faltantes (`dropna`), casteo y tipado de variables, ordenamiento cronológico y redondeo estadístico.

---

### Detalles del Aporte e Implementación Técnica

#### A) Optimización del Patrón de Expresión Regular (`PATRON_LUGAR`)
El texto de origen provisto por la API del USGS sigue el formato estándar: `"15 km ESE of Lima, Peru"`. Un análisis exploratorio determinó que los patrones iniciales fallaban al procesar direcciones con tres puntos cardinales combinados (como **ESE** o **WNW**). 
* **Solución aplicada:** Se modificó el cuantificador de la clase de caracteres a `{1,3}`: `r"(\d+)\s*km\s+([NSEW]{1,3})\s+of\s+([A-Za-zà-ÿ\s]+),\s*Peru"`. Esto garantiza la captura robusta de cualquier azimut o dirección sin sesgar la muestra de datos.

#### B) Flujo de Transformación y Limpieza Indexada
* **Consistencia temporal:** Conversión de la marca de tiempo Unix (en milisegundos) a objetos nativos `datetime` de Pandas mediante `pd.to_datetime(..., unit="ms")`.
* **Depuración estadística:** Eliminación estricta de filas con nulos en variables críticas (`Magnitud` y `Profundidad_km`) mediante `.dropna()`, asegurando que los posteriores análisis vectoriales no generen excepciones.
* **Redondeo Numérico:** Para optimizar el rendimiento y la legibilidad en las visualizaciones finales, las variables de posición geográfica (`Latitud`, `Longitud`) y magnitud sufren un tratamiento de redondeo explícito a decimales significativos.

In [ ]:
import re
import pandas as pd

class ProcesadorSismos:
    """Clase encargada de limpiar, validar y transformar los datos de sismos."""

    # PATRÓN OPTIMIZADO: Captura direcciones compuestas de hasta 3 caracteres (Ej: ESE, NNE)
    PATRON_LUGAR = r"(\d+)\s*km\s+([NSEW]{1,3})\s+of\s+([A-Za-zà-ÿ\s]+),\s*Peru"

    def __init__(self, datos_api: dict):
        self.datos = datos_api
        self.df_limpio = None

    def extraer_info_lugar(self, texto_lugar: str) -> tuple[float | None, str | None, str]:
        """
        Usa expresiones regulares para realizar la minería de texto y extraer
        métricas estructuradas: (distancia_km, direccion, ciudad_referencia).
        """
        if not isinstance(texto_lugar, str):
            return None, None, "Desconocido"
            
        coincidencia = re.match(self.PATRON_LUGAR, texto_lugar.strip())
        if coincidencia:
            distancia_km = float(coincidencia.group(1))
            direccion = coincidencia.group(2)
            ciudad = coincidencia.group(3).strip()
            return distancia_km, direccion, ciudad
        else:
            return None, None, texto_lugar.strip()

    def _clasificar_nivel(self, magnitud: float) -> str:
        """Clasificación auxiliar de magnitudes para segmentar los reportes gráficos."""
        if magnitud < 4.5:
            return "Leve"
        elif magnitud < 5.5:
            return "Moderado"
        else:
            return "Fuerte"

    def transformar_a_dataframe(self) -> pd.DataFrame:
        """Transforma el GeoJSON estructurado de la API en un DataFrame de Pandas depurado."""
        if not self.datos or "features" not in self.datos:
            print("[ERROR] No hay datos válidos para procesar.")
            return pd.DataFrame()

        registros = []
        for sismo in self.datos["features"]:
            props = sismo.get("properties", {})
            geometria = sismo.get("geometry", {})
            coords = geometria.get("coordinates", [None, None, None])

            lugar_texto = props.get("place", "")
            distancia_km, direccion, ciudad = self.extraer_info_lugar(lugar_texto)

            registros.append({
                "Fecha": pd.to_datetime(props.get("time"), unit="ms", errors="coerce"),
                "Magnitud": props.get("mag"),
                "Lugar": lugar_texto,
                "Ciudad_Referencia": ciudad,
                "Distancia_km": distancia_km,
                "Direccion": direccion,
                "Profundidad_km": coords[2],
                "Latitud": coords[1],
                "Longitud": coords[0],
            })

        df = pd.DataFrame(registros)
        
        # Depuración de registros incompletos
        df = df.dropna(subset=["Magnitud", "Profundidad_km"])

        # Mapeo vectorizado de la columna categórica 'Nivel' usando el método interno
        df["Nivel"] = df["Magnitud"].apply(self._clasificar_nivel)

        # Redondeo estadístico y estandarización analítica
        df["Profundidad_km"] = df["Profundidad_km"].round(2)
        df["Latitud"] = df["Latitud"].round(4)
        df["Longitud"] = df["Longitud"].round(4)

        # Ordenamiento cronológico de eventos más recientes a más antiguos
        df = df.sort_values(by="Fecha", ascending=False).reset_index(drop=True)
        self.df_limpio = df
        return self.df_limpio

print("[INFO] Clase ProcesadorSismos documentada y cargada en el Notebook.")